<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# Notebook 01: Dataset Exploration and XGBoost Baseline (bank transfer schema)

This notebook loads the **proprietary transfer extract** (16 feature columns + label),
creates temporal train/val/test splits, and trains a GPU-accelerated **XGBoost baseline**.

**Primary metric**: Average Precision (AUPRC) -- more informative than AUROC at low
positive rates.

## Outline

0. Setup, schema declaration
1. Load the raw extract
2. Temporal split (+ downstream train/eval subsets)
3. Data visualization
4. XGBoost baseline on the 16 raw features
5. Conclusion

---

### The schema, declared once

Every notebook and every tokenizer step reads the column names from `BankSchema`.
**This notebook is the only place you edit them**: it writes
`data/bank/tokenizer/schema.json`, which notebook 02 picks up and embeds in the
saved tokenizer state, so notebooks 03-05 inherit it automatically.

| Column | Role in the model | Tokenizer (notebook 02) |
|---|---|---|
| `customer_id` | sequence key | none (grouping only) |
| `account_number` | account identity within the sequence | `ACCTH` (hash buckets) |
| `datetime` | ordering + inter-transaction gap | `TDIF` (log bins) |
| `counterparty_account` | counterparty identity | `CPTY` (hash buckets) |
| `counterparty_bic` | counterparty bank | `BIC` (frequency vocab) |
| `counterparty_country` | counterparty country | `CTRY` (frequency vocab) |
| `employee` | operator | `EMP` (frequency vocab, ordinal) |
| `comment_structure` | comment type | `CMT` |
| `operations_status` | status flag | `OPST` |
| `movement_code` | transaction type (MCC role) | `MVC` (frequency vocab) |
| `movement_sign` | sign | `SIGN` |
| `dc_flag` | debit/credit | `DC` |
| `month`, `dow`, `hour` | calendar position | `MONTH`, `DOW`, `HOUR` |
| `amt_euro` | amount | `AMT` (quantile bins) |

A customer can hold several accounts. By default one **sequence per customer**
interleaves all of their accounts chronologically (with an `ACCTH` token marking which
account each transaction is on), so an embedding can see behaviour that spans accounts.
`BankTokenizerConfig(sequence_scope="account")` in notebook 02 switches to one sequence
per account instead. Section 3 below prints the history length under both.

In [ ]:
import json
import sys
import time
import warnings
from dataclasses import asdict
from pathlib import Path

warnings.filterwarnings("ignore")

import cudf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import OrdinalEncoder

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.tokenizer import BankSchema

print(f"cuDF version:    {cudf.__version__}")
print(f"XGBoost version: {xgb.__version__}")
print(f"GPU available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU 0:           {props.name}, {props.total_memory / 1e9:.1f} GB")

XGB_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

In [ ]:
# ======================================================================
# EDIT HERE: where the data lives and what the columns are called.
# ======================================================================
from src.experiment import shared_data_dir
# Locations: set them in configs/paths.yaml (the only place), not here.
from src.experiment import raw_file
DATA_DIR = shared_data_dir()
RAW_TABLE = raw_file()                               # .parquet or .csv

SCHEMA = BankSchema(
    customer_id="customer_id",
    account_number="account_number",
    datetime="datetime",
    counterparty_account="counterparty_account",
    counterparty_bic="counterparty_bic",
    counterparty_country="counterparty_country",
    employee="employee",
    comment_structure="comment_structure",
    operations_status="operations_status",
    movement_code="movement_code",
    movement_sign="movement_sign",
    dc_flag="dc_flag",
    month="month",
    dow="dow",
    hour="hour",
    amount="amt_euro",
    description=None,           # free-text column, e.g. "description"; read only if NB02 sets text_strategy
    label="label",
    # aliases={"source_column_name": "canonical_name"},  # bulk renames
)

# Time split: train | val | test by date (the cutoffs are quantiles of time).
TRAIN_RATIO, VAL_RATIO = 0.8, 0.1
# Labelled rows for the downstream (XGBoost) step -- see section 2.1:
#   train: EVERY positive of the train period + NEG_PER_POS random negatives each.
#   val / test: the whole period (EVAL_SAMPLES = None), at its true fraud rate.
#   A number instead keeps that many rows, still at the true rate.
NEG_PER_POS = 50
# Ceiling on the training set's rows (embedding time, host RAM, XGBoost GPU memory:
# ~2 KB per row at 512 dims). Above it negatives are cut first, never below one per
# positive; every positive is kept as long as that allows.
MAX_TRAIN_ROWS = 2_000_000
EVAL_SAMPLES = None
# ======================================================================

SPLIT_DIR = DATA_DIR / "temporal_split"
TOKENIZER_DIR = DATA_DIR / "tokenizer"
OUTPUTS_DIR = DATA_DIR / "outputs"
for d in (SPLIT_DIR, TOKENIZER_DIR, OUTPUTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert RAW_TABLE.exists(), (
    f"Raw extract not found: {RAW_TABLE}\n"
    "Point RAW_TABLE at your parquet/CSV export."
)

(TOKENIZER_DIR / "schema.json").write_text(json.dumps(asdict(SCHEMA), indent=2))

print(f"Raw extract : {RAW_TABLE}")
print(f"Splits      : {SPLIT_DIR}")
print(f"Schema      : {TOKENIZER_DIR / 'schema.json'}")
print(f"Label column: {SCHEMA.label}")
print(f"Features    : {len(SCHEMA.feature_columns)}d -- {SCHEMA.feature_columns}")

---

## 1. Load the Raw Extract

Column names are normalized (lowercase, non-alphanumerics to `_`) and matched against
the schema, so `"Counterparty BIC"` resolves to `counterparty_bic` without any manual
renaming. Anything still unresolved raises immediately with the list of what is missing.

In [ ]:
def read_table(path, columns=None):
    """Read a parquet or CSV extract into cuDF."""
    path = Path(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return cudf.read_parquet(str(path), columns=columns)
    return cudf.read_csv(str(path), usecols=columns)


print("Loading raw extract with cuDF (GPU)...")
t0 = time.time()
raw_gdf = read_table(RAW_TABLE)
print(f"  Load time: {time.time() - t0:.2f}s")

raw_gdf = raw_gdf.rename(columns=SCHEMA.rename_map(list(raw_gdf.columns)))
missing = SCHEMA.missing(list(raw_gdf.columns))
assert not missing, (
    f"Missing required columns: {missing}\n"
    f"Available: {sorted(raw_gdf.columns)}\n"
    "Fix the names in the SCHEMA cell above (or pass aliases=...)."
)
assert SCHEMA.label in raw_gdf.columns, (
    f"Label column '{SCHEMA.label}' not found. Set BankSchema(label=...)."
)

print(f"Shape: {raw_gdf.shape[0]:,} rows x {raw_gdf.shape[1]} columns")
raw_gdf.head(3).to_pandas()

In [ ]:
print("Column summary:")
print(f"{'#':>3} {'column':<24} {'dtype':<18} {'unique':>14} {'nulls':>12}")
print("-" * 76)
for i, col in enumerate(SCHEMA.feature_columns + [SCHEMA.label], start=1):
    series = raw_gdf[col]
    print(
        f"{i:>3} {col:<24} {str(series.dtype):<18} "
        f"{series.nunique():>14,} {int(series.isnull().sum()):>12,}"
    )

In [ ]:
def to_binary_label(series):
    """Coerce a label column (bool / int / 'Yes' / 'true' / ...) to int32 0-1."""
    if series.dtype == "object":
        upper = series.astype("str").str.strip().str.upper()
        return upper.isin(["1", "Y", "YES", "TRUE", "T", "FRAUD"]).astype("int32")
    return (series.fillna(0).astype("float64") > 0).astype("int32")


raw_gdf["_target"] = to_binary_label(raw_gdf[SCHEMA.label])
n_pos = int(raw_gdf["_target"].sum())
print(f"Positives: {n_pos:,} / {len(raw_gdf):,} ({n_pos / len(raw_gdf):.4%})")
print("Raw label values -> binary mapping:")
print(
    cudf.DataFrame({"label": raw_gdf[SCHEMA.label], "_target": raw_gdf["_target"]})
    .drop_duplicates()
    .head(10)
    .to_pandas()
)

---

## 2. Temporal Split

An 80/10/10 split on **time**, cut at day boundaries so a single day never straddles
two splits: the test set holds the most recent transactions, which is the only honest
setup for a sequential model.

Note the residual leakage that any temporal split has: an account active around a
cutoff contributes history to one split and labels to the next. That is intended --
it mirrors production, where the model always scores a customer it has seen before.

Each split gets a `__row_id__` column. Notebook 04 uses it to map the labelled rows
back onto the full split after preprocessing re-orders transactions by account and
time, so **no notebook has to re-derive a sample from a random seed**.

In [ ]:
dt = raw_gdf[SCHEMA.datetime]
if dt.dtype.kind != "M":
    dt = cudf.to_datetime(dt)
    raw_gdf[SCHEMA.datetime] = dt

# Each account's number within its customer (1 = the customer's first account), over the
# full history, before the split, so it is the same in every split and window. Read by the
# tokenizer option account_token: order (ACCT tokens). Also counterparty_order: the same
# number when the counterparty is one of the customer's own accounts, else 0 (option
# own_account_counterparty).
from src.tokenizer.bank_pipeline import add_account_order
raw_gdf = add_account_order(raw_gdf, SCHEMA)
dt = raw_gdf[SCHEMA.datetime]

NS_PER_DAY = 86_400_000_000_000
ts = dt.astype("int64")

train_cut = int(int(ts.quantile(TRAIN_RATIO)) // NS_PER_DAY * NS_PER_DAY)
val_cut = int(int(ts.quantile(TRAIN_RATIO + VAL_RATIO)) // NS_PER_DAY * NS_PER_DAY)

print(f"Date range        : {pd.Timestamp(int(ts.min()))} .. {pd.Timestamp(int(ts.max()))}")
print(f"Train/Val cutoff  : {pd.Timestamp(train_cut)}")
print(f"Val/Test cutoff   : {pd.Timestamp(val_cut)}")

splits = {
    "train": raw_gdf[ts < train_cut],
    "val": raw_gdf[(ts >= train_cut) & (ts < val_cut)],
    "test": raw_gdf[ts >= val_cut],
}

total = sum(len(part) for part in splits.values())
print(f"\n{'Split':<8} {'Rows':>14} {'%':>8} {'Positives':>12} {'Rate':>10}")
print("-" * 56)
for name, part in splits.items():
    pos = int(part["_target"].sum())
    print(
        f"{name:<8} {len(part):>14,} {len(part) / total * 100:>7.2f}% "
        f"{pos:>12,} {pos / max(len(part), 1):>9.4%}"
    )
print("-" * 56)
print(f"{'Total':<8} {total:>14,}")

In [ ]:
for name in list(splits):
    part = splits[name].reset_index(drop=True)
    part["__row_id__"] = np.arange(len(part), dtype="int64")
    splits[name] = part
    path = SPLIT_DIR / f"{name}.parquet"
    part.to_parquet(str(path), index=False)
    print(f"Saved {path} ({len(part):,} rows)")

del raw_gdf

### 2.1 Downstream training and evaluation subsets

Pretraining (notebooks 02-03) uses the **whole** train period without labels --
learning what normal sequences look like is the point of a foundation model. The
supervised step (notebooks 04-05) uses labelled rows of the same periods:

- `train_balanced.parquet` -- **every** positive of the train period plus
  `NEG_PER_POS` random negatives per positive: the XGBoost training set. Nothing caps
  the positives.
- `val_eval.parquet` / `test_eval.parquet` -- by default the **whole** val and test
  periods (`EVAL_SAMPLES = None`), at the true fraud rate: early stopping uses val,
  every reported number comes from test. A number keeps that many rows, still at the
  true rate (fewer positives -> noisier metrics).

Test is the last period and is never seen by pretraining or by any training step.

Undersampling negatives shifts the base rate, so the model's predicted probabilities
are not calibrated to production. That does not affect AUROC or AP, which are ranking
metrics evaluated on the untouched-prevalence subsets. If you need calibrated scores,
refit a calibration layer (e.g. isotonic) on a natural-prevalence sample.

These files are **written once and reused** by notebooks 04 and 05.

In [ ]:
def balanced_sample(gdf, neg_per_pos, max_rows=None, seed=42):
    """Every positive + *neg_per_pos* random negatives per positive, within *max_rows*.

    Undersampling negatives keeps the supervised step affordable. It shifts the
    base rate, so predicted probabilities are not calibrated to production --
    but ROC-AUC and Average Precision are ranking metrics, and they are reported
    on val/test, which keep the true prevalence.
    """
    pos = gdf[gdf["_target"] == 1]
    neg = gdf[gdf["_target"] == 0]
    n_neg = min(len(neg), len(pos) * neg_per_pos)
    if max_rows and len(pos) + n_neg > max_rows:
        # Cut negatives first (but keep at least one per positive), then positives.
        n_neg = min(n_neg, max(max_rows - len(pos), max_rows // 2))
        n_pos = max_rows - n_neg
        print(f"  NOTE: {len(pos):,} positives x {neg_per_pos} negatives exceeds "
              f"MAX_TRAIN_ROWS={max_rows:,}: keeping {min(n_pos, len(pos)):,} positives, "
              f"{n_neg:,} negatives ({n_neg / max(min(n_pos, len(pos)), 1):.1f} per positive)")
        if n_pos < len(pos):
            pos = pos.sample(n=n_pos, random_state=seed)
    parts = [pos]
    if n_neg:
        parts.append(neg if n_neg == len(neg) else neg.sample(n=n_neg, random_state=seed))
    out = cudf.concat(parts, ignore_index=True)
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


def stratified_sample(gdf, n_samples, seed=42):
    """Subsample while preserving the positive rate; None keeps everything."""
    if n_samples is None or n_samples >= len(gdf):
        return gdf.reset_index(drop=True)
    pos = gdf[gdf["_target"] == 1]
    neg = gdf[gdf["_target"] == 0]
    frac = n_samples / len(gdf)
    n_pos = min(len(pos), int(round(len(pos) * frac)))
    n_neg = min(len(neg), n_samples - n_pos)
    parts = []
    if n_pos:
        parts.append(pos.sample(n=n_pos, random_state=seed))
    if n_neg:
        parts.append(neg.sample(n=n_neg, random_state=seed))
    out = cudf.concat(parts, ignore_index=True)
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


train_balanced = balanced_sample(splits["train"], NEG_PER_POS, MAX_TRAIN_ROWS, RANDOM_STATE)
val_eval = stratified_sample(splits["val"], EVAL_SAMPLES, RANDOM_STATE)
test_eval = stratified_sample(splits["test"], EVAL_SAMPLES, RANDOM_STATE)

for name, part in [
    ("train_balanced", train_balanced),
    ("val_eval", val_eval),
    ("test_eval", test_eval),
]:
    path = SPLIT_DIR / f"{name}.parquet"
    part.to_parquet(str(path), index=False)
    pos = int(part["_target"].sum())
    print(
        f"Saved {path.name:<22} {len(part):>9,} rows, "
        f"{pos:>7,} positives ({pos / max(len(part), 1):.4%})"
    )

---

## 3. Data Visualization

In [ ]:
train_gdf = splits["train"]

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# 1. Class imbalance
ax1 = axes[0, 0]
counts = (
    train_gdf["_target"].value_counts().to_pandas().sort_index().reindex([0, 1], fill_value=0)
)
bars = ax1.bar(
    ["Normal", "Positive"], counts.values, color=["#2ca02c", "#d62728"], edgecolor="black"
)
for bar, val in zip(bars, counts.values):
    ax1.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{val:,}",
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )
ax1.set_yscale("log")
ax1.set_ylabel("Transaction count (log scale)")
ax1.set_title("Label distribution (train)", fontsize=13, fontweight="bold")

# 2. Positive rate by movement code (the MCC analogue)
ax2 = axes[0, 1]
MIN_TXNS = 500
by_code = train_gdf.groupby(SCHEMA.movement_code)["_target"].agg(["sum", "count"])
by_code = by_code[by_code["count"] >= MIN_TXNS]
by_code["rate"] = by_code["sum"] / by_code["count"] * 100
top_codes = by_code.sort_values("rate", ascending=False).head(15).to_pandas()
labels = [f"{code} (n={int(row['count']):,})" for code, row in top_codes.iterrows()]
colors = ["#d62728" if r > 1.0 else "#ff7f0e" for r in top_codes["rate"]]
bars = ax2.barh(range(len(top_codes)), top_codes["rate"], color=colors, edgecolor="black")
ax2.set_yticks(range(len(top_codes)))
ax2.set_yticklabels(labels, fontsize=9)
ax2.invert_yaxis()
ax2.set_xlabel("Positive rate (%)")
ax2.set_title(f"Positive rate by movement code (min {MIN_TXNS} txns)", fontsize=13, fontweight="bold")

# 3. History length per sequence -- customer-scoped vs account-scoped
ax3 = axes[1, 0]
per_customer = train_gdf.groupby(SCHEMA.customer_cols).size().to_pandas().values
per_account = train_gdf.groupby(SCHEMA.account_cols).size().to_pandas().values
accounts_per_customer = (
    train_gdf[SCHEMA.account_cols]
    .drop_duplicates()
    .groupby(SCHEMA.customer_cols)
    .size()
    .to_pandas()
    .values
)

p95 = int(np.percentile(per_customer, 95))
ax3.hist(
    np.clip(per_account, 0, p95), bins=50, color="#1f77b4",
    edgecolor="black", alpha=0.6, label="per account",
)
ax3.hist(
    np.clip(per_customer, 0, p95), bins=50, color="#d62728",
    edgecolor="black", alpha=0.5, label="per customer (all accounts)",
)
ax3.axvline(
    np.median(per_customer), color="red", linestyle="--", linewidth=1.5,
    label=f"Median per customer: {int(np.median(per_customer)):,}",
)
ax3.set_title("History length per sequence (train)", fontsize=13, fontweight="bold")
ax3.set_xlabel(f"Transactions (clipped at p95 = {p95:,})")
ax3.set_ylabel("Sequences")
ax3.legend(fontsize=9)

# 4. Volume over time with split boundaries
ax4 = axes[1, 1]
monthly_frames = []
for name, part in splits.items():
    d = part[SCHEMA.datetime]
    ym = (d.dt.year * 100 + d.dt.month).rename("ym")
    counts = ym.to_frame().groupby("ym").size().rename("count").reset_index()
    counts["split"] = name
    monthly_frames.append(counts.to_pandas())
monthly = pd.concat(monthly_frames).sort_values("ym").reset_index(drop=True)
palette = {"train": "#2ca02c", "val": "#1f77b4", "test": "#d62728"}
for name in ("train", "val", "test"):
    part = monthly[monthly["split"] == name]
    ax4.plot(part.index, part["count"].values, color=palette[name], linewidth=1.6, label=name)
ax4.set_title("Monthly volume with temporal splits", fontsize=13, fontweight="bold")
ax4.set_ylabel("Transactions per month")
step = max(1, len(monthly) // 12)
ax4.set_xticks(list(monthly.index[::step]))
ax4.set_xticklabels([str(v) for v in monthly["ym"].values[::step]], rotation=45)
ax4.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "dataset_overview.png", dpi=150)
plt.show()
print(f"Saved -> {OUTPUTS_DIR / 'dataset_overview.png'}")

print("\nSequence length depends on what one sequence is (notebook 02, "
      "BankTokenizerConfig.sequence_scope):")
print(f"  accounts per customer     : median {int(np.median(accounts_per_customer))}, "
      f"max {int(accounts_per_customer.max())}, "
      f"{(accounts_per_customer > 1).mean():.1%} of customers have more than one")
print(f"  transactions per account  : median {int(np.median(per_account)):,}, "
      f"p95 {int(np.percentile(per_account, 95)):,}")
print(f"  transactions per customer : median {int(np.median(per_customer)):,}, "
      f"p95 {int(np.percentile(per_customer, 95)):,}")
print("  Customer-scoped sequences (the default) interleave a customer's accounts "
      "chronologically, so each embedding can see cross-account behaviour; the "
      "model gets an ACCTH token to tell the accounts apart.")

---

## 4. XGBoost Baseline (GPU-accelerated)

16 raw features, no hand-crafted aggregates: this is the number the foundation model
embeddings have to beat in notebook 05. `datetime` enters as an epoch-seconds column
(`_ts`) instead of a string, since an ordinal-encoded timestamp string is meaningless.

High-cardinality identifiers (`customer_id`, `account_number`, `counterparty_account`)
are ordinal-encoded. That is deliberately naive -- it is exactly the modelling gap the
sequence model closes.

In [ ]:
FEATURE_COLS = [c for c in SCHEMA.feature_columns if c != SCHEMA.datetime] + ["_ts"]


def to_frame(gdf):
    """cuDF split -> (pandas features, numpy labels)."""
    out = gdf[[c for c in SCHEMA.feature_columns if c != SCHEMA.datetime]].copy()
    out["_ts"] = (gdf[SCHEMA.datetime].astype("int64") // 1_000_000_000)
    y = gdf["_target"].to_pandas().values
    return out[FEATURE_COLS].to_pandas(), y


X_train, y_train = to_frame(train_balanced)
X_val, y_val = to_frame(val_eval)
X_test, y_test = to_frame(test_eval)

print(f"Feature matrix: {len(FEATURE_COLS)} columns")
print(f"  Train: {X_train.shape}, positives {y_train.sum():,} ({y_train.mean():.4%})")
print(f"  Val:   {X_val.shape}, positives {y_val.sum():,} ({y_val.mean():.4%})")
print(f"  Test:  {X_test.shape}, positives {y_test.sum():,} ({y_test.mean():.4%})")

preprocessor = make_column_transformer(
    (
        OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        make_column_selector(dtype_include=["object", "category"]),
    ),
    remainder="passthrough",
)
X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc = preprocessor.transform(X_val)
X_test_enc = preprocessor.transform(X_test)
print(f"Encoded: train {X_train_enc.shape}, val {X_val_enc.shape}, test {X_test_enc.shape}")

In [ ]:
XGB_PARAMS = {
    "n_estimators": 400,
    "max_depth": 8,
    "learning_rate": 0.05,
    "colsample_bytree": 0.95,
    "min_child_weight": 12,
    "subsample": 0.673,
    "reg_alpha": 0.01,
    "reg_lambda": 0.001,
    "random_state": RANDOM_STATE,
}


def train_xgb(X_tr, y_tr, X_va, y_va, X_te, y_te, params, name="Model"):
    """Train XGBoost with early stopping; report AUROC and Average Precision."""
    print(f"\nTraining {name}...")
    print(f"  Features: {X_tr.shape[1]}d | Samples: {X_tr.shape[0]:,}")
    t0 = time.time()
    clf = xgb.XGBClassifier(
        **params,
        scale_pos_weight=1.0,
        tree_method="hist",
        device=XGB_DEVICE,
        early_stopping_rounds=20,
        eval_metric="auc",
    )
    clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    train_time = time.time() - t0

    val_preds = clf.predict_proba(X_va)[:, 1]
    test_preds = clf.predict_proba(X_te)[:, 1]
    metrics = {
        "val_auc": roc_auc_score(y_va, val_preds),
        "val_ap": average_precision_score(y_va, val_preds),
        "test_auc": roc_auc_score(y_te, test_preds),
        "test_ap": average_precision_score(y_te, test_preds),
    }
    print(f"  Train time: {train_time:.1f}s (best_iteration={clf.best_iteration})")
    print(f"  Val  ROC-AUC: {metrics['val_auc']:.4f} | AP: {metrics['val_ap']:.4f}")
    print(f"  Test ROC-AUC: {metrics['test_auc']:.4f} | AP: {metrics['test_ap']:.4f}")
    return clf, metrics


clf_baseline, metrics_baseline = train_xgb(
    X_train_enc, y_train, X_val_enc, y_val, X_test_enc, y_test,
    params=XGB_PARAMS,
    name=f"XGBoost baseline ({len(FEATURE_COLS)}d raw features)",
)

(OUTPUTS_DIR / "baseline_metrics.json").write_text(
    json.dumps(
        {
            "n_features": len(FEATURE_COLS),
            "feature_cols": FEATURE_COLS,
            "train_rows": int(len(X_train)),
            "params": XGB_PARAMS,
            "metrics": {k: float(v) for k, v in metrics_baseline.items()},
        },
        indent=2,
    )
)
print(f"\nSaved baseline metrics -> {OUTPUTS_DIR / 'baseline_metrics.json'}")

In [ ]:
try:
    encoded_names = [
        n.split("__", 1)[-1] for n in preprocessor.get_feature_names_out()
    ]
except Exception:  # older sklearn: object columns first, then passthrough
    obj_cols = [c for c in X_train.columns if str(X_train[c].dtype) in ("object", "category")]
    encoded_names = obj_cols + [c for c in X_train.columns if c not in obj_cols]

importances = pd.Series(
    clf_baseline.feature_importances_, index=encoded_names
).sort_values(ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
ax1.bar(
    ["Test AUROC", "Test AUPRC"],
    [metrics_baseline["test_auc"], metrics_baseline["test_ap"]],
    color=["#1f77b4", "#ff7f0e"], edgecolor="black",
)
for i, val in enumerate([metrics_baseline["test_auc"], metrics_baseline["test_ap"]]):
    ax1.text(i, val + 0.01, f"{val:.4f}", ha="center", va="bottom", fontweight="bold")
ax1.set_ylim(0, 1.05)
ax1.set_title("NB01 baseline (test set)", fontsize=13, fontweight="bold")

importances.plot.barh(ax=ax2, color="#1f77b4", edgecolor="black")
ax2.set_title("Feature importance (gain)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "baseline_metrics.png", dpi=150)
plt.show()

---

## 5. Conclusion

The baseline uses 16 raw columns with no aggregate features and no history: each row is
scored in isolation. Note the AUROC/AUPRC gap -- at a low positive rate, AUROC saturates
while Average Precision still has room, which is why AP is the metric to watch.

**Outputs**

| File | Used by |
|---|---|
| `temporal_split/{train,val,test}.parquet` | NB02 (pretraining corpus), NB04 (history) |
| `temporal_split/train_balanced.parquet` | NB04, NB05 (supervised training set) |
| `temporal_split/{val,test}_eval.parquet` | NB04, NB05 (evaluation subsets) |
| `tokenizer/schema.json` | NB02 (column names) |
| `outputs/baseline_metrics.json` | NB05 (lift reference) |

Continue to [02_seq_preproc_tokenization.ipynb](./02_seq_preproc_tokenization.ipynb).